<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/selective-decoder-routing/blob/main/notebooks/00_dataset_and_feasibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Selective Decoder Routing — Dataset and Feasibility Analysis

**Week 1 | Incremental Colab notebook**

**Dataset:** [Bauxitiego/surface-code-syndromes](https://huggingface.co/datasets/Bauxitiego/surface-code-syndromes) — a reformatted version of experimental surface-code data from Google Quantum AI.

**Research question:** How often could choosing correlated matching instead of PyMatching correct an otherwise incorrect logical prediction?

Each row is an experimental shot containing a syndrome, the true logical `label`, and **stored predictions** from `pymatching` and `correlated_matching`. We do not run either decoder during this feasibility study.

**Scope:** Use the published **training** split for Week 1 development. The published validation split is reserved for model/threshold selection; the **test split must remain untouched** until final evaluation. This notebook does not train ML or QML models. Ground-truth labels and stored decoder predictions may be used to create routing targets and evaluate decoders, but must **never** become router input features.

## 1. Minimal setup

Download only the Python modules, configuration, and tests needed for this notebook from **one resolved GitHub commit**. No repository clone, model download, QML package, or `pip install -e .` is needed for this small subset. The original implementation remains in `src/router/`; do not copy its calculations into notebook cells.

For final reproducibility, replace `SOURCE_REF = "main"` with the resolved commit SHA printed below.

In [1]:
%pip install -q datasets numpy pandas PyYAML

In [2]:
import json
import os
import sys
from pathlib import Path
from urllib.request import urlopen

REPO = "rifahnanjiba02-arch/selective-decoder-routing"
SOURCE_REF = "main"  # Use the printed commit SHA for a frozen final run.
ROOT = Path("/content/selective_decoder_week1")

with urlopen(f"https://api.github.com/repos/{REPO}/commits/{SOURCE_REF}") as response:
    repo_commit = json.load(response)["sha"]

required_files = [
    "configs/data.yaml",
    "src/router/__init__.py",
    "src/router/data.py",
    "src/router/decoders.py",
    "src/router/labels.py",
    "src/router/routing.py",
    "tests/test_data.py",
    "tests/test_routing.py",
]

for relative_path in required_files:
    destination = ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://raw.githubusercontent.com/{REPO}/{repo_commit}/{relative_path}"
    with urlopen(url) as response:
        destination.write_bytes(response.read())

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print(f"Loaded {len(required_files)} project files at commit: {repo_commit}")

Loaded 8 project files at commit: 2cdae59baf5628c2c48b87d4934c8ae3de0ddc42


## 2. Run settings

The team configuration currently leaves `dataset_revision: null`. To prevent dataset changes from silently altering this notebook's sample, use the **verified Hugging Face dataset commit** below in a *temporary local copy* of the config. The repository's original config is not modified.

In [3]:
import yaml

DATASET_REVISION = "9c51b27ffc51ed7ff3ef3a72167ebf4862d1f2f3"

with open(ROOT / "configs/data.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

assert config["dataset_name"] == "Bauxitiego/surface-code-syndromes"
assert config.get("dataset_revision") in (None, DATASET_REVISION)

pinned_config = dict(config)
pinned_config["dataset_revision"] = DATASET_REVISION
config_path = ROOT / "configs/data_colab_pinned.yaml"
config_path.write_text(yaml.safe_dump(pinned_config, sort_keys=False), encoding="utf-8")

run_settings = {
    "repository_commit": repo_commit,
    "dataset": pinned_config["dataset_name"],
    "dataset_revision": DATASET_REVISION,
    "split": "train",
    "sample_size": pinned_config["development"]["train_size"],
    "sampling_seed": pinned_config["sampling_seed"],
    "published_split_seed": pinned_config["source_split_seed"],
}
for name, value in run_settings.items():
    print(f"{name}: {value}")

repository_commit: 2cdae59baf5628c2c48b87d4934c8ae3de0ddc42
dataset: Bauxitiego/surface-code-syndromes
dataset_revision: 9c51b27ffc51ed7ff3ef3a72167ebf4862d1f2f3
split: train
sample_size: 50000
sampling_seed: 42
published_split_seed: 20260806


## 3. Existing offline unit tests

Run the teammate-written tests before loading real data. These tests check function behavior on small inputs; they **do not** establish results on the real experimental dataset.

In [4]:
import os
import sys
import subprocess
from pathlib import Path

ROOT = Path("/content/selective_decoder_week1")

# Check that the setup downloaded the required files
assert (ROOT / "tests").is_dir(), "Run the setup cell first."
assert (ROOT / "src/router").is_dir(), "Router files are missing."

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT / "src")

subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", "tests", "-v"],
    cwd=ROOT,
    env=env,
    check=True
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], returncode=0)

## 4. Load the real training data

Call the **existing** `load_surface_code_dataset()` and `sample_split()` functions. The loader requests the `train` split and the notebook selects rows only from `train`. This run's Hugging Face builder also downloaded/prepared validation and test files while loading; no validation or test rows were selected for analysis. The full training split is prepared before the 50,000-shot sample is selected, so this is not a streaming sample.

Published dataset-card split sizes: train **4,550,000**, validation **650,000**, test **1,300,000**. This notebook directly confirms the train count; the validation and test counts are taken from the card.

In [5]:
from router.data import load_surface_code_dataset, sample_split

splits, loaded_config = load_surface_code_dataset(
    config_path=str(config_path), split_name="train"
)

sample = sample_split(
    splits,
    split_name="train",
    sample_size=loaded_config["development"]["train_size"],
    seed=loaded_config["sampling_seed"],
)

assert len(sample) == run_settings["sample_size"]
print("Full training split:", len(splits["train"]))
print("Development sample:", len(sample))
print("Columns:", sample.column_names)

README.md:   0%|          | 0.00/6.90k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B / 69.9MB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 10.0MB            

validation.parquet: downloading bytes:           |  0.00B            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Full training split: 4550000
Development sample: 50000
Columns: ['experiment', 'basis', 'distance', 'rounds', 'time_steps', 'width', 'round_widths', 'syndrome', 'label', 'pymatching', 'correlated_matching', 'tensor_network']


In [6]:
# Check that the same seed reproduces the sample's stored labels/predictions.
repeat = sample_split(
    splits,
    split_name="train",
    sample_size=run_settings["sample_size"],
    seed=run_settings["sampling_seed"],
)
for column in ("label", "pymatching", "correlated_matching"):
    assert list(sample[column]) == list(repeat[column])
print("Repeated sampling agrees on all stored label/prediction columns.")

Repeated sampling agrees on all stored label/prediction columns.


In [7]:
# Compact preview: do not print full syndrome arrays.
from IPython.display import display

preview_columns = [
    "experiment", "basis", "distance", "rounds",
    "label", "pymatching", "correlated_matching",
]
display(sample.select(range(5)).select_columns(preview_columns).to_pandas())

,experiment,basis,distance,rounds,label,pymatching,correlated_matching
0,surface_code_bZ_d3_r13_center_5_7,Z,3,13,1,1,1
1,surface_code_bZ_d3_r03_center_5_3,Z,3,3,0,0,0
2,surface_code_bX_d3_r01_center_3_5,X,3,1,0,0,0
3,surface_code_bZ_d5_r13_center_5_5,Z,5,13,0,0,0
4,surface_code_bZ_d5_r03_center_5_5,Z,5,3,0,0,0


## 5. Complete the training-sample inspection

The saved Colab outputs below check **train rows only**. The results are summarized after the cells and in the [data dictionary](../docs/DATA_DICTIONARY.md).


In [8]:
# Compare all fields in small batches instead of copying all syndromes at once.
assert len(sample) == len(repeat)
for start in range(0, len(sample), 256):
    stop = min(start + 256, len(sample))
    assert sample[start:stop] == repeat[start:stop], f"Rows {start}:{stop} differ"
full_row_repeat_ok = True
print(f"All {len(sample):,} rows match in the same order.")

All 50,000 rows match in the same order.


### Observed value sets

In [9]:
import pandas as pd

INSPECT_FIELDS = ("label", "pymatching", "correlated_matching", "basis", "distance", "rounds")
EXPECTED = {
    "label": {0, 1}, "pymatching": {0, 1}, "correlated_matching": {0, 1},
    "basis": {"X", "Z"}, "distance": {3, 5}, "rounds": set(range(1, 26, 2)),
}


def missing_scalar(value):
    return value is None or bool(pd.isna(value))


value_rows = []
unexpected_counts = {}
for field in INSPECT_FIELDS:
    present = [value for value in sample[field] if not missing_scalar(value)]
    distinct = sorted(set(present), key=lambda value: (type(value).__name__, value))
    unexpected_counts[field] = sum(value not in EXPECTED[field] for value in present)
    value_rows.append({
        "field": field, "type": str(sample.features[field]),
        "observed_values": ", ".join(repr(value) for value in distinct),
        "unexpected_rows": unexpected_counts[field],
    })
display(pd.DataFrame(value_rows))

,field,type,observed_values,unexpected_rows
0,label,Value('uint8'),"0, 1",0
1,pymatching,Value('uint8'),"0, 1",0
2,correlated_matching,Value('uint8'),"0, 1",0
3,basis,Value('string'),"'X', 'Z'",0
4,distance,Value('int8'),"3, 5",0
5,rounds,Value('int16'),"1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25",0


### Missing counts and data types

In [10]:
missing_counts = {}
missing_rows = []
for field in sample.column_names:
    values = sample[field]
    # Sequence contents are checked in the next cell.
    missing = sum(value is None for value in values) if field in {"syndrome", "round_widths"} else sum(missing_scalar(value) for value in values)
    missing_counts[field] = missing
    missing_rows.append({"field": field, "type": str(sample.features[field]), "missing_rows": missing})
display(pd.DataFrame(missing_rows))
print("Unexpected scalar values:", unexpected_counts)

,field,type,missing_rows
0,experiment,Value('string'),0
1,basis,Value('string'),0
2,distance,Value('int8'),0
3,rounds,Value('int16'),0
4,time_steps,Value('int8'),0
5,width,Value('int16'),0
6,round_widths,List(Value('uint8')),0
7,syndrome,List(Value('uint8')),0
8,label,Value('uint8'),0
9,pymatching,Value('uint8'),0


Unexpected scalar values: {'label': 0, 'pymatching': 0, 'correlated_matching': 0, 'basis': 0, 'distance': 0, 'rounds': 0}


### Syndrome / `round_widths` shape checks

In [11]:
CHECKS = (
    "invalid_dimensions", "time_steps_not_rounds_plus_one", "syndrome_length_mismatch",
    "round_widths_length_mismatch", "invalid_round_width", "nonbinary_syndrome",
    "nonzero_padding",
)
shape_counts = {name: 0 for name in CHECKS}
shape_examples = {name: [] for name in CHECKS}
padded_positions = nonzero_padded_positions = 0


def flag(name, index):
    shape_counts[name] += 1
    if len(shape_examples[name]) < 5:
        shape_examples[name].append(index)


for index, row in enumerate(sample):
    steps, width, rounds = row["time_steps"], row["width"], row["rounds"]
    syndrome, widths = row["syndrome"], row["round_widths"]
    if any(not isinstance(value, int) or isinstance(value, bool) for value in (steps, width, rounds)) or steps <= 0 or width <= 0:
        flag("invalid_dimensions", index)
        continue
    if steps != rounds + 1:
        flag("time_steps_not_rounds_plus_one", index)
    syndrome_ok = isinstance(syndrome, (list, tuple)) and len(syndrome) == steps * width
    widths_ok = isinstance(widths, (list, tuple)) and len(widths) == steps
    if not syndrome_ok:
        flag("syndrome_length_mismatch", index)
    if not widths_ok:
        flag("round_widths_length_mismatch", index)
    if isinstance(syndrome, (list, tuple)) and any(value not in (0, 1) for value in syndrome):
        flag("nonbinary_syndrome", index)
    if not widths_ok:
        continue
    if any(not isinstance(real, int) or isinstance(real, bool) or not 0 <= real <= width for real in widths):
        flag("invalid_round_width", index)
        continue
    if not syndrome_ok:
        continue
    bad_padding = False
    for step, real in enumerate(widths):
        padding = syndrome[step * width + real : (step + 1) * width]
        padded_positions += len(padding)
        nonzero = sum(value != 0 for value in padding)
        nonzero_padded_positions += nonzero
        bad_padding |= nonzero > 0
    if bad_padding:
        flag("nonzero_padding", index)

display(pd.DataFrame([
    {"check": name, "affected_rows": shape_counts[name], "example_sample_indices": shape_examples[name]}
    for name in CHECKS
]))
print(f"Padded positions checked: {padded_positions:,}; nonzero padded positions: {nonzero_padded_positions:,}")

,check,affected_rows,example_sample_indices
0,invalid_dimensions,0,[]
1,time_steps_not_rounds_plus_one,0,[]
2,syndrome_length_mismatch,0,[]
3,round_widths_length_mismatch,0,[]
4,invalid_round_width,0,[]
5,nonbinary_syndrome,0,[]
6,nonzero_padding,0,[]


Padded positions checked: 514,200; nonzero padded positions: 0


### Unresolved schema notes

In [12]:
# Copy this factual summary into docs/DATA_DICTIONARY.md after reviewing it.
report = [
    "## Week 1 training-sample checks",
    f"- Dataset revision: `{run_settings['dataset_revision']}`",
    f"- Train sample: {len(sample):,} rows, seed {run_settings['sampling_seed']}",
    f"- Full-row repeat check: {full_row_repeat_ok}",
    "", "| Field | Type | Observed values | Missing rows | Unexpected rows |",
    "| --- | --- | --- | ---: | ---: |",
]
for row in value_rows:
    field = row["field"]
    report.append(f"| `{field}` | {row['type']} | {row['observed_values']} | {missing_counts[field]} | {unexpected_counts[field]} |")
report += ["", "| Shape/padding check | Affected rows |", "| --- | ---: |"]
report += [f"| `{name}` | {shape_counts[name]} |" for name in CHECKS]
report.append(f"- Nonzero padded positions: {nonzero_padded_positions}")
report.append("- Other missing counts: " + ", ".join(
    f"`{field}`={count}" for field, count in missing_counts.items() if field not in INSPECT_FIELDS
))
print("\n".join(report))
if any(unexpected_counts.values()) or any(shape_counts.values()) or any(missing_counts[field] for field in INSPECT_FIELDS):
    print("Investigate and document the anomalies before marking Task 1 complete.")
else:
    print("Sample checks passed. Record the summary in docs/DATA_DICTIONARY.md.")

## Week 1 training-sample checks
- Dataset revision: `9c51b27ffc51ed7ff3ef3a72167ebf4862d1f2f3`
- Train sample: 50,000 rows, seed 42
- Full-row repeat check: True

| Field | Type | Observed values | Missing rows | Unexpected rows |
| --- | --- | --- | ---: | ---: |
| `label` | Value('uint8') | 0, 1 | 0 | 0 |
| `pymatching` | Value('uint8') | 0, 1 | 0 | 0 |
| `correlated_matching` | Value('uint8') | 0, 1 | 0 | 0 |
| `basis` | Value('string') | 'X', 'Z' | 0 | 0 |
| `distance` | Value('int8') | 3, 5 | 0 | 0 |
| `rounds` | Value('int16') | 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25 | 0 | 0 |

| Shape/padding check | Affected rows |
| --- | ---: |
| `invalid_dimensions` | 0 |
| `time_steps_not_rounds_plus_one` | 0 |
| `syndrome_length_mismatch` | 0 |
| `round_widths_length_mismatch` | 0 |
| `invalid_round_width` | 0 |
| `nonbinary_syndrome` | 0 |
| `nonzero_padding` | 0 |
- Nonzero padded positions: 0
- Other missing counts: `experiment`=0, `time_steps`=0, `width`=0, `round_widths`=0, `s

### Measured Task 1 results (saved Colab run, 24 October 2026)

- Dataset revision: `9c51b27ffc51ed7ff3ef3a72167ebf4862d1f2f3`; `train` contains 4,550,000 rows.
- Sample: 50,000 train rows with seed `42`. Repeating the selection matched **every field of every sampled row in the same order**.
- All 12 expected columns were present. No missing rows were found in the 11 fields used for this analysis; `tensor_network` was null in all 50,000 sampled rows and is outside the two-decoder scope.

| Field | Dataset type | Observed values | Missing rows | Unexpected rows |
| --- | --- | --- | ---: | ---: |
| `label` | `Value('uint8')` | 0, 1 | 0 | 0 |
| `pymatching` | `Value('uint8')` | 0, 1 | 0 | 0 |
| `correlated_matching` | `Value('uint8')` | 0, 1 | 0 | 0 |
| `basis` | `Value('string')` | 'X', 'Z' | 0 | 0 |
| `distance` | `Value('int8')` | 3, 5 | 0 | 0 |
| `rounds` | `Value('int16')` | 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25 | 0 | 0 |

- On all 50,000 rows, `time_steps == rounds + 1`, `len(syndrome) == time_steps * width`, and `len(round_widths) == time_steps`. Every real round width was within `0..width`, and all syndrome values were binary.
- Checked 514,200 padded positions: **0** nonzero values. Every recorded shape/padding issue count was **0**.
- The [data dictionary](../docs/DATA_DICTIONARY.md) records the full field list, missing counts, and these checks. No decoder outcome counts or routing-target results are included here yet.

## 6. Decoder outcome results

### Four-count table

### Five-rate table

## 7. Routing target

## 8. Checks and interpretation

## 9. Compact findings note — `results/feasibility/`